# Phân tích xác suất thắng map VALORANT sau từng round

Notebook này đi theo đúng luồng của dự án: đọc dữ liệu VCT, kiểm tra chất lượng, tạo một mẫu sau mỗi round, chia dữ liệu theo thời gian và đọc kết quả mô hình.

Phạm vi thí nghiệm chính là VCT 2023–2024. Năm 2023 được chia thành train/validation theo loạt trận; toàn bộ năm 2024 chỉ dùng làm test cuối. Raw CSV không nằm trong GitHub public.

## Dữ liệu dùng trong bài

| Bảng | Cột chính | Ý nghĩa trong bài |
|---|---|---|
| `maps_scores.csv` | `Team A`, `Team B`, điểm cuối | Tạo nhãn cuối map và kiểm tra chuỗi round. Điểm cuối không đưa vào đầu vào mô hình. |
| `eco_rounds.csv` | `Round Number`, `Loadout Value`, `Remaining Credits`, `Type`, `Outcome` | Mô tả trạng thái kinh tế và kết quả của mỗi đội trong từng round. |
| `overview.csv` | `Team`, `Player`, `Agents`, `Side` | Lấy đội hình 5 đặc vụ ở dòng `Side = both`. Các chỉ số kill/rating cuối map bị bỏ để tránh nhìn thấy tương lai. |

Mỗi mẫu sau xử lý có lịch sử nhiều round. Nhãn là `1` nếu Team A thắng cả map, `0` nếu Team B thắng. Đây là phân tích hồi cứu sau trận, không phải dự đoán trực tiếp trong trận.

In [ ]:
# Path giúp nối đường dẫn mà không phải tự ghép dấu /.
from pathlib import Path
import os
import pandas as pd

# Khi chạy từ thư mục notebooks, parent là thư mục gốc của repository.
# Path.cwd() lấy thư mục mà Jupyter đang mở notebook.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# Có thể trỏ tới thư mục dữ liệu khác bằng biến môi trường này.
# Ưu tiên biến môi trường để notebook chạy được ở máy khác mà không sửa code.
DATA_ROOT = Path(os.environ.get(
    "VALORANT_DATA_ROOT",
    REPO_ROOT.parent / "RawData" / "Valorant-Champion-Tour-2021-2026-Data",
))
# assert dừng sớm với thông báo dễ hiểu nếu chưa trỏ đúng thư mục RawData.
assert DATA_ROOT.is_dir(), f"Không tìm thấy dữ liệu: {DATA_ROOT}"
print(DATA_ROOT)

In [ ]:
# Đọc ba bảng cho hai mùa để thấy quy mô dữ liệu gốc.
# Đọc từng mùa độc lập để thấy quy mô gốc trước khi làm sạch.
for season in (2023, 2024):
    # Mỗi mùa có thư mục matches chứa ba bảng dùng trong bài.
    folder = DATA_ROOT / f"vct_{season}" / "matches"
    # maps_scores giữ đội và điểm cuối map; không dùng điểm cuối làm input.
    scores = pd.read_csv(folder / "maps_scores.csv", low_memory=False)
    # eco_rounds có hai dòng cho mỗi round, một dòng cho mỗi đội.
    economy = pd.read_csv(folder / "eco_rounds.csv", low_memory=False)
    # overview có Agent và các thống kê sau map; bài chỉ lấy đội hình Agent.
    overview = pd.read_csv(folder / "overview.csv", low_memory=False)
    print({
        "season": season,
        "maps_scores_rows": len(scores),
        "economy_rows": len(economy),
        "overview_rows": len(overview),
        "economy_missing": economy[["Loadout Value", "Remaining Credits", "Type", "Outcome"]].isna().sum().to_dict(),
    })

## Ghép hai dòng kinh tế thành một round

Trong file economy, một round có hai dòng: một dòng cho Team A và một dòng cho Team B. Mã nguồn trong `src/data.py` xác định phía dựa trên tên đội, đổi `3.6k` thành `3600`, rồi đưa hai dòng lên cùng một hàng.

Sau đó mã nguồn kiểm tra ba điều: round phải liên tục từ 1 đến round cuối; tổng số round thắng phải khớp điểm cuối trong `maps_scores.csv`; mỗi đội phải có đủ 5 đặc vụ. Map không đạt một trong các điều kiện này bị loại cả map, không tự điền dữ liệu.

In [ ]:
# Thêm thư mục gốc vào sys.path để notebook gọi được src/data.py.
import sys
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_prefix_samples, chronological_split, load_prepared_data

# Hàm này ghép, lọc và kiểm tra ba bảng thành một dòng cho mỗi round hợp lệ.
round_table = load_prepared_data(DATA_ROOT, seasons=(2023, 2024))
# Mỗi map tạo ra một mẫu sau round 3, 4,... nhưng bỏ round cuối.
samples = build_prefix_samples(round_table, min_round=3)
# Tách theo series: 2023 chia train/validation, 2024 giữ nguyên làm test.
split = chronological_split(samples, validation_fraction=0.2, seed=42)

print("Số dòng round sau kiểm tra:", len(round_table))
print("Số mẫu sau round 3 trở đi:", len(samples))
print("Số map theo tập:", split.map_counts)
pd.DataFrame(samples[:3])[["season", "map", "team_A", "team_B", "round", "score_A", "score_B", "target"]]

## Mô hình và cách đọc kết quả

Ba mô hình ML là Logistic Regression, XGBoost và CatBoost. Chúng nhận trạng thái hiện tại, trung bình lịch sử, thay đổi từ round đầu, map và tập đặc vụ đã mã hóa.

Mô hình DL chính là GRU. GRU đọc lần lượt trạng thái round nên giữ được thông tin chuỗi, ví dụ một đội vừa thắng liên tiếp hay economy thay đổi ra sao. Có thêm một phiên bản gộp đội hình bằng attention để so sánh; attention không được xem là bằng chứng đặc vụ gây ra thắng thua.

`Log Loss` và `Brier` đo chất lượng xác suất, càng thấp càng tốt. `ROC-AUC` đo khả năng xếp hạng hai lớp, càng cao càng tốt. Vì mục tiêu là xác suất thắng, không nên chỉ báo cáo accuracy.

In [ ]:
# Sau khi chạy scripts/train.py, file này chứa metric và dữ liệu dẫn xuất cho web.
import json
from IPython.display import display

# demo.json là kết quả dẫn xuất nhỏ; CSV gốc vẫn nằm ngoài repo public.
demo_path = REPO_ROOT / "web" / "data" / "demo.json"
if demo_path.is_file():
    result = json.loads(demo_path.read_text(encoding="utf-8"))
    metrics = pd.DataFrame([
        {
            "model": item["model"],
            **item["test"],
        }
        for item in result["models"]
    ]).sort_values("log_loss")
    display(metrics.round(4))
else:
    print("Chưa có demo.json. Chạy: /opt/homebrew/opt/python@3.13/libexec/bin/python3 scripts/train.py")

## Kết luận khi chạy lần hiện tại

Kết quả không được ghi cứng trong notebook vì có thể thay đổi nếu phiên bản thư viện hoặc dữ liệu thay đổi. Hãy lấy bảng metric và biểu đồ được tạo từ lần chạy thật để điền vào báo cáo.

Lưu ý quan trọng: đây là quan hệ dự đoán trong dữ liệu lịch sử. Không được diễn giải rằng loadout hoặc một đặc vụ tự nó gây ra xác suất thắng; muốn kết luận nhân quả cần thiết kế nghiên cứu khác.